Extract all main campus zip codes and Fall 2023 enrollment data for public and private non-profit 4+ year universities.

In [17]:
# Imports
import os
import pyodbc
import getpass
import pandas as pd
import numpy as np
from pathlib import Path
from rapidfuzz import process, fuzz
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

In [16]:
# Set SerpAPI Key

os.environ["SERP_API_KEY"] = getpass.getpass("Enter your API key: ")

In [2]:
# Paths

DB_PATH = Path("data/IPEDS_23-24/IPEDS202324.accdb")
OUT_CSV = Path("data/university_enrollment_address.csv")

# Helpers

def format_ipeds_zip(value):
    """
    IPEDS ZIP fields may be stored without a dash.
    Examples:
      60637      -> 60637
      060102301  -> 06010-2301
    Returns a normalized string or None.
    """
    if pd.isna(value):
        return None

    s = str(value).strip()

    # remove trailing .0 if Access/pandas coerces to float
    if s.endswith(".0"):
        s = s[:-2]

    # keep digits only
    s = "".join(ch for ch in s if ch.isdigit())

    if not s:
        return None

    # 5-digit ZIP
    if len(s) <= 5:
        return s.zfill(5)

    # 9-digit ZIP+4
    if len(s) == 9:
        return f"{s[:5]}-{s[5:]}"

    # anything else: preserve raw digits
    return s


def get_access_connection(db_path: Path):
    conn_str = (
        r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
        f"DBQ={db_path};"
    )
    return pyodbc.connect(conn_str)


# Main

def main():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"Database not found: {DB_PATH}")

    conn = get_access_connection(DB_PATH)

    try:
        # EF2023 has multiple records per institution.
        # EFLEVEL = 10 corresponds to "All students total".
        query = """
        SELECT
            h.UNITID,
            h.INSTNM,
            h.ADDR,
            h.CITY,
            h.STABBR,
            h.ZIP,
            e.EFTOTAL
        FROM
            HD2023 AS h
            INNER JOIN EF2023 AS e
                ON h.UNITID = e.UNITID
        WHERE
            e.EFLEVEL = 10
            AND h.SECTOR IN (1, 2)
        """

        df = pd.read_sql(query, conn)

    finally:
        conn.close()

    # clean up columns
    df = df.rename(columns={
        "INSTNM": "institution_name",
        "ADDR":   "street_address",
        "CITY":   "city",
        "STABBR": "state",
        "ZIP":    "zip_raw",
        "EFTOTAL": "fall_2023_total_enrollment"
    })

    # normalize ZIP
    df["zip_code"] = df["zip_raw"].apply(format_ipeds_zip)

    # optional split fields
    df["zip5"] = df["zip_code"].str.extract(r"^(\d{5})", expand=False)
    df["zip4"] = df["zip_code"].str.extract(r"^\d{5}-(\d{4})$", expand=False)

    # full address string
    df["full_address"] = (
        df["street_address"] + ", " +
        df["city"] + ", " +
        df["state"] + " " +
        df["zip_code"].fillna("")
    ).str.strip()

    # tidy final column order
    df = df[
        [
            "UNITID",
            "institution_name",
            "fall_2023_total_enrollment",
            "full_address",
            "street_address",
            "city",
            "state",
            "zip_code",
            "zip5",
            "zip4",
            "zip_raw",
        ]
    ].sort_values(["institution_name", "UNITID"]).reset_index(drop=True)

    df.to_csv(OUT_CSV, index=False)

    print(f"Saved {len(df):,} rows to: {OUT_CSV.resolve()}")
    print("\nPreview:")
    print(df.head(20).to_string(index=False))


if __name__ == "__main__":
    main()

C:\Users\jwram\AppData\Local\Temp\ipykernel_25812\1300386784.py:80: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Saved 2,427 rows to: C:\Users\jwram\.ipython\university_caffeine_analysis\data\university_enrollment_address.csv

Preview:
 UNITID                                                              institution_name  fall_2023_total_enrollment                                         full_address                  street_address                 city state   zip_code  zip5 zip4    zip_raw
 177834                                       A T Still University of Health Sciences                        3629                800 W Jefferson, Kirksville, MO 63501                 800 W Jefferson           Kirksville    MO      63501 63501  NaN      63501
 180203                                                        Aaniiih Nakoda College                         133               269 Blackfeet Avenue, Harlem, MT 59526            269 Blackfeet Avenue               Harlem    MT      59526 59526  NaN      59526
 222178                                                  Abilene Christian University               

In [ ]:
# Assign the top ~100 universities according to U.S. News

top_100 = {
    1: "Princeton University",                                    2: "Massachusetts Institute of Technology",
    3: "Harvard University",                                      4: "Stanford University",
    5: "Yale University",                                         6: "University of Chicago",
    7: "Duke University",                                         8: "Johns Hopkins University",
    9: "Northwestern University",                                10: "University of Pennsylvania",
   11: "California Institute of Technology",                     12: "Cornell University",
   13: "Brown University",                                       14: "Dartmouth College",
   15: "Columbia University in the City of New York",            16: "University of California, Berkeley",
   17: "Rice University",                                        18: "University of California, Los Angeles",
   19: "Vanderbilt University",                                  20: "Carnegie Mellon University",
   21: "University of Michigan, Ann Arbor",                      22: "University of Notre Dame",
   23: "Washington University in St. Louis",                     24: "Emory University",
   25: "Georgetown University",                                  26: "University of North Carolina at Chapel Hill",
   27: "University of Virginia, Main Campus",                    28: "University of Southern California",
   29: "University of California, San Diego",                    30: "University of Florida",
   31: "The University of Texas at Austin",                      32: "Georgia Institute of Technology, Main Campus",
   33: "New York University",                                    34: "University of California, Davis",
   35: "University of California, Irvine",                       36: "Boston College",
   37: "Tufts University",                                       38: "University of Illinois Urbana-Champaign",
   39: "University of Wisconsin-Madison",                        40: "University of California, Santa Barbara",
   41: "Ohio State University, Main Campus",                     42: "Boston University",
   43: "Rutgers University-New Brunswick",                       44: "University of Maryland, College Park",
   45: "University of Washington, Seattle Campus",               46: "Lehigh University",
   47: "Northeastern University",                                48: "Purdue University, Main Campus",
   49: "University of Georgia",                                  50: "University of Rochester",
   51: "Case Western Reserve University",                        52: "Florida State University",
   53: "Texas A & M University, College Station",                54: "Virginia Polytechnic Institute and State University",
   55: "Wake Forest University",                                 56: "College of William and Mary",
   57: "University of California, Merced",                       58: "Villanova University",
   59: "George Washington University",                           60: "Pennsylvania State University, University Park",
   61: "Santa Clara University",                                 62: "Stony Brook University",
   63: "University of Minnesota, Twin Cities",                   64: "Michigan State University",
   65: "North Carolina State University at Raleigh",             66: "Rensselaer Polytechnic Institute",
   67: "University of Massachusetts Amherst",                    68: "University of Miami",
   69: "Brandeis University",                                    70: "Tulane University of Louisiana",
   71: "University of Connecticut",                              72: "University of Pittsburgh, Pittsburgh Campus",
   73: "Binghamton University",                                  74: "Indiana University-Bloomington",
   75: "Clemson University",                                     76: "Rutgers University-Newark",
   77: "Syracuse University",                                    78: "University at Buffalo",
   79: "University of California, Riverside",                    80: "Colorado School of Mines",
   81: "Drexel University",                                      82: "New Jersey Institute of Technology",
   83: "Stevens Institute of Technology",                        84: "Pepperdine University",
   85: "University of Illinois Chicago",                         86: "Worcester Polytechnic Institute",
   87: "Yeshiva University",                                     88: "American University",
   89: "Baylor University",                                      90: "Howard University",
   91: "Marquette University",                                   92: "Rochester Institute of Technology",
   93: "Southern Methodist University",                          94: "University of California, Santa Cruz",
   95: "University of Delaware",                                 96: "University of South Florida",
   97: "Florida International University",                       98: "Fordham University",
   99: "Rutgers University-Camden",                             100: "Texas Christian University",
}

In [11]:
# Extract likely matches for T100 in IPEDS
df = pd.read_csv("data/university_enrollment_address.csv")
ipeds_names = df["institution_name"].tolist()

matched_rows = []
ambiguous = []
unmatched = []

for rank, target in top_100.items():
    # get top 3 fuzzy matches
    hits = process.extract(target, ipeds_names, scorer=fuzz.token_sort_ratio, limit=3)
    best_name, best_score, _ = hits[0]

    if best_score < 70:
        unmatched.append((rank, target))
        print(f"[NO MATCH]   #{rank} {target}")
        continue

    # check if second match is close to first (ambiguous)
    if len(hits) > 1 and hits[1][1] >= best_score - 5:
        ambiguous.append((rank, target, hits))
        print(f"[AMBIGUOUS]  #{rank} {target}")
        for name, score, _ in hits:
            print(f"               {score:.0f}  {name}")
        continue

    matched_row = df[df["institution_name"] == best_name].iloc[0].to_dict()
    matched_row["usnews_rank"] = rank
    matched_row["usnews_name"] = target
    matched_rows.append(matched_row)

top_100_df = pd.DataFrame(matched_rows).sort_values("usnews_rank").reset_index(drop=True)

print(f"\nMatched: {len(matched_rows)} / {len(top_100)}")
print(f"Ambiguous: {len(ambiguous)}, Unmatched: {len(unmatched)}")

[AMBIGUOUS]  #9 Northwestern University
               100  Northwestern University
               96  Northeastern University
               93  Northwest University
[AMBIGUOUS]  #16 University of California, Berkeley
               80  University of California-Los Angeles
               75  University of California-Davis
               75  University of California-Santa Barbara
[AMBIGUOUS]  #29 University of California, San Diego
               87  University of California-San Diego
               84  Dominican University of California
               80  University of California-Davis
[AMBIGUOUS]  #41 Ohio State University, Main Campus
               84  Ohio State University-Lima Campus
               84  Ohio State University-Main Campus
               81  Ohio State University-Marion Campus
[AMBIGUOUS]  #44 University of Maryland, College Park
               76  University of Maryland-College Park
               75  Notre Dame of Maryland University
               75  University o

In [ ]:
# Hardcode ambiguous matches
overrides = {
     9: "Northwestern University",
    16: "University of California-Berkeley",
    29: "University of California-San Diego",
    41: "Ohio State University-Main Campus",
    44: "University of Maryland-College Park",
    45: "University of Washington-Seattle Campus",
    47: "Northeastern University",
    53: "Texas A & M University-College Station",
    56: "William & Mary",
    60: "Pennsylvania State University-Main Campus",
    67: "University of Massachusetts-Amherst",
    94: "University of California-Santa Cruz",
}

for rank, ipeds_name in overrides.items():
    match = df[df["institution_name"] == ipeds_name]
    if match.empty:
        print(f"[OVERRIDE NOT FOUND] #{rank} -> {ipeds_name}")
        continue
    matched_row = match.iloc[0].to_dict()
    matched_row["usnews_rank"] = rank
    matched_row["usnews_name"] = top_100[rank]
    matched_rows.append(matched_row)

top_100_df = pd.DataFrame(matched_rows).sort_values("usnews_rank").reset_index(drop=True)
print(f"Final: {len(top_100_df)} / {len(top_100)}")

Final: 100 / 100


In [13]:
top_100_df.head()

,UNITID,institution_name,fall_2023_total_enrollment,full_address,street_address,city,state,zip_code,zip5,zip4,zip_raw,usnews_rank,usnews_name
0,186131,Princeton University,8922,"1 Nassau Hall, Princeton, NJ 08544-0070",1 Nassau Hall,Princeton,NJ,08544-0070,8544,70.0,08544-0070,1,Princeton University
1,166683,Massachusetts Institute of Technology,11920,"77 Massachusetts Avenue, Cambridge, MA 02139-4307",77 Massachusetts Avenue,Cambridge,MA,02139-4307,2139,4307.0,02139-4307,2,Massachusetts Institute of Technology
2,166027,Harvard University,30386,"Massachusetts Hall, Cambridge, MA 02138",Massachusetts Hall,Cambridge,MA,02138,2138,NaN,02138,3,Harvard University
3,243744,Stanford University,18446,", Stanford, CA 94305",,Stanford,CA,94305,94305,NaN,94305,4,Stanford University
4,130794,Yale University,15081,"Woodbridge Hall, New Haven, CT 06520",Woodbridge Hall,New Haven,CT,06520,6520,NaN,06520,5,Yale University


In [ ]:
# Convert addresses to coordinates

# geocoder setup
geolocator = Nominatim(user_agent="university_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# geocode each address
lats, lons, confirmed = [], [], []

for _, row in top_100_df.iterrows():
    query = f"{row['street_address']}, {row['city']}, {row['state']} {row['zip5']}"
    
    try:
        location = geocode(query)
        if location:
            lats.append(location.latitude)
            lons.append(location.longitude)
            confirmed.append(location.address)
            print(f"✓ {row['institution_name']}\n  -> {location.address}\n")
        else:
            lats.append(None)
            lons.append(None)
            confirmed.append(None)
            print(f"✗ {row['institution_name']} — no result\n")
    except Exception as e:
        lats.append(None)
        lons.append(None)
        confirmed.append(None)
        print(f"✗ {row['institution_name']} — {e}\n")

top_100_df["lat"] = lats
top_100_df["lon"] = lons
top_100_df["geocoded_address"] = confirmed

top_100_df.to_csv("data/top_100.csv", index=False)
print(f"saved {len(top_100_df)} rows to data/top_100.csv")

✗ Princeton University — no result

✓ Massachusetts Institute of Technology
  -> School of Architecture and Planning, 77, Massachusetts Avenue, Cambridge, Middlesex County, Massachusetts, 02139, United States

✓ Harvard University
  -> Massachusetts Hall, 11, Harvard Yard, Cambridge, Middlesex County, Massachusetts, 02138, United States

✓ Stanford University
  -> Stanford, Santa Clara County, California, 94305, United States

✓ Yale University
  -> Woodbridge Hall, 109, Wall Street, Downtown, Dixwell, New Haven, South Central Connecticut Planning Region, Connecticut, 06511, United States

✓ University of Chicago
  -> The University of Chicago, 5801, South Ellis Avenue, Hyde Park, Chicago, Hyde Park Township, Cook County, Illinois, 60637, United States

✗ Duke University — no result

✓ Johns Hopkins University
  -> REMINGTON AVE & 31ST ST fs nb, 3400, North Charles Street, Charles Village, Baltimore, Maryland, 21218, United States

✓ Northwestern University
  -> Northwestern University

In [ ]:
# Query and store SerpAPI cafe results per-university address

In [ ]:
# pip install pandas geopandas osmnx shapely pyogrio fiona

from pathlib import Path
import re
import time
import pandas as pd
import geopandas as gpd
import osmnx as ox

# Paths

UNIV_CSV = Path("data/university_enrollment_zipcode.csv")
ZCTA_SHP = Path("data/tl_2020_us_zcta520/tl_2020_us_zcta520.shp")
OUT_CSV = Path("caffeine_establishments.csv")
OUT_CSV_AG = Path("caffeine_establishments_overlap.csv")

# harccoded brand whitelist
BRAND_PATTERNS = [
    r"\bstarbucks\b",
    r"\bdunkin\b",
    r"\bdunkin'? donuts\b",
    r"\bpeet'?s\b",
    r"\bcaribou coffee\b",
    r"\btim hortons\b",
    r"\bbiggby\b",
    r"\bthe coffee bean\b",
    r"\bhuman bean\b",
    r"\bscooter'?s coffee\b",
    r"\bphilz\b",
    r"\bblue bottle\b",
    r"\bcoffee bean & tea leaf\b",
]

# tags to ask Overpass/OSM for
# OSMnx interprets values as OR within each key
OSM_TAGS = {
    "amenity": ["cafe"],
    "shop": ["coffee"],
}


# Helpers

def norm_zip(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = "".join(ch for ch in s if ch.isdigit())
    if not s:
        return None
    return s[:5].zfill(5)

def has_explicit_caffeine_signal(row):
    """
    Keep:
      - amenity=cafe
      - shop=coffee
      - explicit coffee-chain brands/names
    Reject:
      - broad fast food unless explicitly whitelisted by brand/name
    """
    vals = []
    for col in ["name", "brand", "brand:wikidata", "amenity", "shop", "cuisine"]:
        if col in row and pd.notna(row[col]):
            vals.append(str(row[col]).lower())
    blob = " | ".join(vals)

    # direct structural includes
    if "amenity" in row and str(row.get("amenity", "")).lower() == "cafe":
        return True
    if "shop" in row and str(row.get("shop", "")).lower() == "coffee":
        return True

    # explicit brand include
    return any(re.search(pat, blob) for pat in BRAND_PATTERNS)


# Load Uni CSV

df = pd.read_csv(UNIV_CSV)

# try a few likely zip column names
zip_col = None
for candidate in ["zip_code", "ZIP", "zip", "zipcode", "zip5"]:
    if candidate in df.columns:
        zip_col = candidate
        break

if zip_col is None:
    raise ValueError("Could not find a ZIP column in the university CSV.")

df["zip5"] = df[zip_col].map(norm_zip)
df = df[df["zip5"].notna()].copy()


# Load ZCTA Polygons

zcta = gpd.read_file(ZCTA_SHP)

# 2020 Census ZCTA shapefiles typically use ZCTA5CE20
if "ZCTA5CE20" not in zcta.columns:
    raise ValueError("Expected ZCTA5CE20 in the ZCTA shapefile.")

zcta["zip5"] = zcta["ZCTA5CE20"].astype(str).str.zfill(5)

needed_zips = sorted(df["zip5"].dropna().unique())
zcta_sub = zcta[zcta["zip5"].isin(needed_zips)].copy()

# project to lat/lon for Overpass
if zcta_sub.crs is None:
    zcta_sub = zcta_sub.set_crs(4269)  # TIGER often comes as NAD83
zcta_sub = zcta_sub.to_crs(4326)


# Query OSM / Overpass per ZIP polygon

ox.settings.timeout = 180
ox.settings.use_cache = True
ox.settings.log_console = True

results = []

for i, row in zcta_sub.iterrows():
    zip5 = row["zip5"]
    geom = row.geometry

    try:
        gdf = ox.features_from_polygon(geom, tags=OSM_TAGS)

        if gdf.empty:
            caffeine_count = 0
        else:
            work = gdf.reset_index(drop=False).copy()

            # flatten columns to plain strings if needed
            work.columns = [str(c) for c in work.columns]

            # keep only explicit caffeine-related places
            mask = work.apply(has_explicit_caffeine_signal, axis=1)
            keep = work[mask].copy()

            # dedupe aggressively: same OSM element only once
            dedupe_cols = [c for c in ["element", "id", "name", "brand", "geometry"] if c in keep.columns]
            if dedupe_cols:
                keep = keep.drop_duplicates(subset=dedupe_cols)
            else:
                keep = keep.drop_duplicates()

            caffeine_count = len(keep)

        results.append({"zip5": zip5, "caffeine_places_zip": caffeine_count})
        print(f"{zip5}: {caffeine_count}")

    except Exception as e:
        print(f"Failed on ZIP {zip5}: {e}")
        results.append({"zip5": zip5, "caffeine_places_zip": pd.NA})

    time.sleep(0.8) # increase or decrease, I need to check the documentation

counts = pd.DataFrame(results)

# Save Outputs

# make sure output parent dirs exist if needed
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_CSV_AG.parent.mkdir(parents=True, exist_ok=True)

# pure ZIP-level counts

zip_counts = counts.copy()

zip_counts = (
    zip_counts
    .dropna(subset=["zip5"])
    .drop_duplicates(subset=["zip5"])
    .sort_values("zip5")
    .reset_index(drop=True)
)

zip_counts.to_csv(OUT_CSV, index=False)

print(f"\nSaved ZIP-level caffeine counts: {OUT_CSV.resolve()}")
print(zip_counts.head(20).to_string(index=False))

# overlap summary when multiple universities share ZIPs

name_col = "institution_name" if "institution_name" in df.columns else "INSTNM"
enroll_col = (
    "fall_2023_total_enrollment"
    if "fall_2023_total_enrollment" in df.columns
    else "EFTOTAL"
)

zip_overlap = (
    df.dropna(subset=["zip5"])
      .groupby("zip5", as_index=False)
      .agg(
          university_count=("UNITID", "nunique"),
          total_enrollment=(enroll_col, "sum"),
          universities=(name_col, lambda s: " | ".join(sorted(set(map(str, s)))))
      )
)

zip_overlap = (
    zip_counts
    .merge(zip_overlap, on="zip5", how="left")
    .sort_values(["university_count", "total_enrollment", "zip5"], ascending=[False, False, True])
    .reset_index(drop=True)
)

zip_overlap.to_csv(OUT_CSV_AG, index=False)

print(f"\nSaved ZIP overlap summary: {OUT_CSV_AG.resolve()}")
print(zip_overlap.head(20).to_string(index=False))

33442: 1
Failed on ZIP 33827: No matching features. Check query location, tags, and log.
